### Load silver tabless

In [0]:
dbutils.widgets.text("catalog_name", "23283_dev")

In [0]:
from pyspark.sql.functions import *

In [0]:
orders = spark.table(f"{catalog_name}.silver_23283.orders")
order_items = spark.table(f"{catalog_name}.silver_23283.order_items")
customers = spark.table(f"{catalog_name}.silver_23283.customers")
products = spark.table(f"{catalog_name}.silver_23283.products")

In [0]:
orders = spark.table(f"{catalog_name}.silver_23283.orders")
order_items = spark.table(f"{catalog_name}.silver_23283.order_items")
customers = spark.table(f"{catalog_name}.silver_23283.customers")
products = spark.table(f"{catalog_name}.silver_23283.products")

In [0]:
from pyspark.sql.functions import col

fact_sales = orders.alias("o") \
    .join(order_items.alias("oi"), col("o.order_id") == col("oi.order_id")) \
    .select(
        col("o.order_id"),
        col("o.customer_id"),
        col("oi.product_id"),
        col("o.order_date"),
        col("o.order_status"),
        col("oi.quantity"),
        col("oi.price"),
        (col("oi.quantity") * col("oi.price")).alias("line_amount"),
        col("o.total_amount"),
        col("o.created_at"),
        col("o.updated_at")
    )

In [0]:
fact_sales.writeTo(
    f"{catalog_name}.gold_23283.fact_sales"
).createOrReplace()

### Dim tables

In [0]:
customers.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog_name}.gold_23283.dim_customers")

products.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog_name}.gold_23283.dim_products")

### Aggregations

In [0]:
from pyspark.sql.functions import col

orders = spark.table(f"{catalog_name}.silver_23283.orders") \
    .withColumn("total_amount", col("total_amount").cast("double"))

order_items = spark.table(f"{catalog_name}.silver_23283.order_items") \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("quantity", col("quantity").cast("int"))

In [0]:
fact_sales = orders.alias("o") \
    .join(order_items.alias("oi"), "order_id") \
    .selectExpr(
        "o.order_id",
        "o.customer_id",
        "oi.product_id",
        "o.order_date",
        "oi.quantity",
        "oi.price",
        "oi.quantity * oi.price as line_amount"
    )

revenue by state

In [0]:
customers = spark.table(f"{catalog_name}.silver_23283.customers")

df = fact_sales.join(customers, "customer_id")

df.groupBy("state") \
  .sum("line_amount") \
  .show()

top products

In [0]:
fact_sales.groupBy("product_id") \
    .sum("quantity") \
    .orderBy("sum(quantity)", ascending=False) \
    .show()

sales trend

In [0]:
from pyspark.sql.functions import to_date

fact_sales.withColumn("order_day", to_date("order_date")) \
    .groupBy("order_day") \
    .sum("line_amount") \
    .show()

Broadcast Joins

In [0]:
from pyspark.sql.functions import broadcast

fact_df = orders.join(
    order_items, "order_id"
).join(
    broadcast(customers), "customer_id"
).join(
    broadcast(products), "product_id"
)

partition

In [0]:
from pyspark.sql.functions import col

fact_sales = fact_sales \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("quantity", col("quantity").cast("int")) \
    .withColumn("line_amount", col("line_amount").cast("double"))

In [0]:
order_items = spark.table(f"{catalog_name}.silver_23283.order_items") \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("quantity", col("quantity").cast("int"))

In [0]:
df = df.coalesce(2)

In [0]:
df = df.repartition("order_date")  # smart partitioning